# SciTS Forecasting with OLMo-1B: Smoke Run and Checkpoint Evaluation

This notebook follows the [SciTS forecasting tutorial](../../docs/tutorials/timeseries/README.md) on an Aurora **debug node**:

1. Check the [environment](../../docs/tutorials/timeseries/environment.md) and the [forecast data](../../docs/tutorials/timeseries/data.md).
2. Run the `PRISM-OLMO-1B-SCITS-FORECAST` [smoke job](../../docs/tutorials/timeseries/training.md) directly on this node.
3. [Evaluate](../../docs/tutorials/timeseries/evaluation.md) the `step_2000` checkpoint from the full run on the held-out `val_shards/`.

The model combines a frozen **OLMo-1B** trunk, the frozen pretrained **Intern-S2 Preview 397B time-series encoder**, a trainable input projector, and PRISM's trainable linear quantile forecast head (720 steps × 10 channels × quantiles 0.1/0.5/0.9).

## 0. Start the kernel on a debug node

The kernel must run **on the compute node** with the pinned Aurora modules loaded and the Intern-S2 venv active, so that PyTorch sees the XPUs and Transformers ≥ 5.2 is importable. From a UAN:

```bash
qsub -I -q debug -l select=1 -l walltime=01:00:00 -l filesystems=flare:home -A <project>

# On the compute node
cd /path/to/PRISM
module purge
module use /opt/aurora/26.26.0/modulefiles
module load oneapi/release/2025.3.1
module load mpich/prd/5.0.0.aurora_test.51a9474
module load frameworks/2025.3.1
export VENV_PATH=/flare/GenesisHackathonOct26/$USER/prism-envs/py3.12-intern-s2
source "$VENV_PATH/bin/activate"
# Packages under ~/.local can shadow the frameworks XPU build of PyTorch.
export PYTHONNOUSERSITE=1
export HF_HOME=/flare/GenesisHackathonOct26/scits/huggingface
export PRISM_DATA_ROOT=/flare/GenesisHackathonOct26/scits/data

jupyter lab --no-browser --ip="$(hostname)" --port 8888
```

The venv is the one built on the [environment page](../../docs/tutorials/timeseries/environment.md) with `tools/build_aurora_env.sh --intern-s2`. It does not include Jupyter; install it once from a login node with `uv pip install jupyterlab ipywidgets` while the venv is active (`ipywidgets` lets progress bars render). The model cache, extracted encoder, SciTS-forecast data, and trained `step_2000` checkpoint are already staged under `/flare/GenesisHackathonOct26/scits`.

Forward the port from your laptop (`ssh -L 8888:<compute-node>:8888 <user>@aurora.alcf.anl.gov` through the UAN), then open this notebook. The smoke job needs the whole node (12 XPUs), so run the evaluation section after it finishes.

## 1. Environment

Set the paths used by the rest of the notebook. Values already exported in the shell take precedence over the defaults below.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").is_file() and (p / "src").is_dir()
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

SHARED_ROOT = Path("/flare/GenesisHackathonOct26/scits")
os.environ.setdefault("HF_HOME", str(SHARED_ROOT / "huggingface"))
os.environ.setdefault("PRISM_DATA_ROOT", str(SHARED_ROOT / "data"))
os.environ.setdefault("VENV_PATH", sys.prefix)
# Compute nodes have no outbound network; read models from the shared cache only.
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

DATA_DIR = Path(os.environ["PRISM_DATA_ROOT"]) / "SciTS-forecast"
ENCODER_CKPT = Path(os.environ["HF_HOME"]) / "intern-s2-preview-397b-timeseries" / "model.safetensors"
SMOKE_ID = "OLMO-SCITS-FORECAST-SMOKE"
MODEL_PRESET = REPO_ROOT / "src/conf/model/prism_olmo1b_intern_s2_397b_ts_forecast.yaml"
EVAL_RUN_DIR = SHARED_ROOT / "checkpoints/PRISM-OLMO-1B-SCITS-FORECAST"
EVAL_CHECKPOINT = EVAL_RUN_DIR / "step_2000"

for name in ("HF_HOME", "PRISM_DATA_ROOT", "VENV_PATH"):
    print(f"{name:16s} {os.environ[name]}")
print(f"{'REPO_ROOT':16s} {REPO_ROOT}")

Confirm that this is a compute node, that PyTorch sees the XPUs, and that the Intern-S2 Transformers variant is installed. The check runs in a subprocess so the kernel does not hold an XPU context while the smoke job runs.

In [ ]:
%%bash
echo "host: $(hostname)   PBS_JOBID: ${PBS_JOBID:-<unset>}"
echo "python: $(which python)"
python - <<'EOF'
import torch, transformers
print(f"torch {torch.__version__} | xpu available: {torch.xpu.is_available()} | devices: {torch.xpu.device_count()}")
print(f"transformers {transformers.__version__} (Intern-S2 needs >= 5.2.0)")
EOF

In [ ]:
required = {
    "Intern-S2 397B time-series encoder": ENCODER_CKPT,
    "OLMo-1B in the shared HF hub": Path(os.environ["HF_HOME"]) / "hub" / "models--allenai--OLMo-1B-0724-hf",
    "SciTS-forecast manifest": DATA_DIR / "manifest.json",
    "SciTS-forecast train shards": DATA_DIR / "shards",
    "SciTS-forecast val shards": DATA_DIR / "val_shards",
    "step_2000 checkpoint": EVAL_CHECKPOINT / "model.safetensors",
}
for label, path in required.items():
    print(f"[{'ok' if path.exists() else 'MISSING'}] {label}: {path}")

## 2. Data

The [data page](../../docs/tutorials/timeseries/data.md) converts SciTS into `SciTS-forecast/` with `shards/`, `val_shards/`, and `manifest.json`. Each forecast sample has a history (`.ts.npy`), a numeric target (`.gt_ts.npy`), its validity mask (`.gt_ts_mask.npy`), prompt text, and metadata.

In [ ]:
import json

manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print({k: v for k, v in manifest.items() if not isinstance(v, (list, dict))})
for split in ("shards", "val_shards"):
    print(f"{split:11s} {len(list((DATA_DIR / split).glob('*.tar')))} tar files")

## 3. Smoke run

The design `PRISM-OLMO-1B-SCITS-FORECAST` in `experiments/LLM_timeseries_scaling.yaml` trains the projector and forecast head with masked pinball loss (batch 2 per rank × 16 accumulation steps × 12 ranks = 384). The smoke run caps training at 50 steps and saves checkpoints at steps 25 and 50.

Because we are already on a debug node, the launcher is used **without `--batch`**: it writes an interactive job script under `jobs/` and runs it here with `mpiexec` on all 12 XPUs. First preview the script with `--dry-run`.

In [ ]:
%%bash
python tools/launch_aurora_web.py \
    --id OLMO-SCITS-FORECAST-SMOKE \
    --file experiments/LLM_timeseries_scaling.yaml \
    --design PRISM-OLMO-1B-SCITS-FORECAST \
    --nodes 1 \
    --dist-strategy hsdp \
    --fsdp-sharding shard_grad_op \
    --max-steps 50 \
    --webdataset-dir "$PRISM_DATA_ROOT/SciTS-forecast" \
    --webdataset-modality time_series \
    --use-shared-venv \
    --hf-home "$HF_HOME" \
    --shared-hf-home "$HF_HOME/hub" \
    training.warmup_steps=5 \
    training.eval_every_n_steps=25 \
    training.save_every_n_steps=25 \
    --dry-run

Check the generated script for the expected data, venv, and output paths.

In [ ]:
import re

job_script = max(
    (REPO_ROOT / "jobs").glob(f"run_aurora_web_{SMOKE_ID}_*_interactive.sh"),
    key=lambda p: p.stat().st_mtime,
)
print(job_script, "\n")
script_text = job_script.read_text()
for line in script_text.splitlines():
    if re.match(r"\s*export (WEBDATASET_\w+|LOCAL_SHARDS_DIR|VENV_PATH|HF_HOME)=", line):
        print(line.strip())
for pattern in (r"hydra\.run\.dir=\S+", r"training\.max_steps=\S+", r"model=\S+"):
    # The design value comes first and the CLI override last; the last one wins.
    print(re.findall(pattern, script_text)[-1])

Run the smoke job. Training logs are verbose, so they go to a file; follow progress from a terminal on the same node with `tail -f logs/olmo_scits_forecast_smoke.log`. Expect several minutes for model staging, data staging, and 50 steps.

In [ ]:
%%bash
mkdir -p logs
python tools/launch_aurora_web.py \
    --id OLMO-SCITS-FORECAST-SMOKE \
    --file experiments/LLM_timeseries_scaling.yaml \
    --design PRISM-OLMO-1B-SCITS-FORECAST \
    --nodes 1 \
    --dist-strategy hsdp \
    --fsdp-sharding shard_grad_op \
    --max-steps 50 \
    --webdataset-dir "$PRISM_DATA_ROOT/SciTS-forecast" \
    --webdataset-modality time_series \
    --use-shared-venv \
    --hf-home "$HF_HOME" \
    --shared-hf-home "$HF_HOME/hub" \
    training.warmup_steps=5 \
    training.eval_every_n_steps=25 \
    training.save_every_n_steps=25 \
    > logs/olmo_scits_forecast_smoke.log 2>&1
echo "launcher exit code: $?"
tail -n 30 logs/olmo_scits_forecast_smoke.log

### Check the smoke result

A healthy smoke run reaches step 50, logs finite losses, and writes `step_25` and `step_50` checkpoints under `outputs/OLMO-SCITS-FORECAST-SMOKE/<date>/<time>/checkpoints/`.

In [ ]:
import math

import matplotlib.pyplot as plt


def read_losses(perf_path):
    """Return (steps, losses) from the trainer's per-log records in perf.jsonl."""
    steps, losses = [], []
    with open(perf_path) as f:
        for line in f:
            record = json.loads(line)
            if record.get("site") == "trainer_native_per_log" and record.get("loss") is not None:
                steps.append(record["step"])
                losses.append(record["loss"])
    return steps, losses


smoke_dir = max((REPO_ROOT / "outputs" / SMOKE_ID).glob("*/*"), key=lambda p: p.stat().st_mtime)
smoke_ckpts = sorted(p.name for p in (smoke_dir / "checkpoints").glob("step_*"))
smoke_steps, smoke_losses = read_losses(smoke_dir / "checkpoints" / "perf.jsonl")

print("run dir:    ", smoke_dir)
print("checkpoints:", smoke_ckpts)
print("last step:  ", smoke_steps[-1] if smoke_steps else None)
print("all finite: ", all(math.isfinite(x) for x in smoke_losses))

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(smoke_steps, smoke_losses, marker=".")
ax.set(xlabel="step", ylabel="training loss", title="Smoke run")
plt.show()

### Full run

The full run is submitted from a UAN, not from this debug node. Use the same command with `--id OLMO-SCITS-FORECAST`, add `--batch --queue capacity --walltime 06:00:00`, and drop `--max-steps 50`, the three `training.*` smoke overrides, and `--dry-run`. It saves a checkpoint every 500 steps. The rest of this notebook evaluates `step_2000` from such a run.

## 4. Evaluate the `step_2000` checkpoint

As explained on the [evaluation page](../../docs/tutorials/timeseries/evaluation.md), the trainer's periodic `eval` records repeat the running training loss, and the text evaluator ignores `.gt_ts.npy` / `.gt_ts_mask.npy`. This section therefore scores the forecast head directly: it runs the checkpoint on held-out `val_shards/` and compares the predicted quantiles with the numeric targets, using only cells where the target mask is valid.

First, the training loss of the full run up to the evaluated checkpoint:

In [ ]:
run_steps, run_losses = read_losses(EVAL_RUN_DIR / "perf.jsonl")
eval_step = int(EVAL_CHECKPOINT.name.split("_")[-1])

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(run_steps, run_losses, lw=0.8)
ax.axvline(eval_step, color="tab:red", ls="--", label=EVAL_CHECKPOINT.name)
ax.set(xlabel="step", ylabel="training loss", title=EVAL_RUN_DIR.name)
ax.legend()
plt.show()

### Load the model

Build the model from the same preset the design trains with, `prism_olmo1b_intern_s2_397b_ts_forecast`, then load the `step_2000` weights the same way `tools/universal_evaluator.py` does.

In [ ]:
import torch
from omegaconf import OmegaConf
from safetensors.torch import load_file
from src.config import ModelConfig
from src.decoders import remap_legacy_decoder_keys
from src.model import UnifiedTransformer
from tools.universal_evaluator import _align_checkpoint_backbone_vocab
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()  # keep weight-loading progress out of the cell output

preset = OmegaConf.to_container(OmegaConf.load(MODEL_PRESET), resolve=True)
preset["llm_backbone_id"] = preset.pop("backbone_id")
preset["llm_tokenizer_id"] = preset.pop("tokenizer_id")
model_config = ModelConfig(**preset)

DEVICE = "xpu:0" if torch.xpu.is_available() else "cpu"
model = UnifiedTransformer(model_config)
state_dict = {
    key.removeprefix("module."): value
    for key, value in load_file(str(EVAL_CHECKPOINT / "model.safetensors")).items()
}
state_dict = remap_legacy_decoder_keys(_align_checkpoint_backbone_vocab(model, state_dict))
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"loaded {len(state_dict)} tensors; missing={len(missing)} unexpected={len(unexpected)}")
model = model.to(DEVICE).eval()
tokenizer = model.backbone_tokenizer

decoder = model.decoders["time_series"]
HORIZON, NUM_VARS = decoder.horizon, decoder.num_vars
QUANTILES = list(decoder.quantiles)
MEDIAN = decoder._median_idx
print(f"device={DEVICE} horizon={HORIZON} num_vars={NUM_VARS} quantiles={QUANTILES}")

### Held-out data

Read `val_shards/` once, in order, with the same loader the trainer uses. It caps history length, decodes the target and mask, and computes the per-channel history mean and scale that the forecast head uses to de-normalize its output.

In [ ]:
from src.data.collate import MultimodalCollator
from src.data.multi_webdataset import ModalityAwareWebDatasetWrapper

val_dataset = ModalityAwareWebDatasetWrapper(
    tokenizer=tokenizer,
    modalities=["time_series"],
    local_shards_dir=str(DATA_DIR / "val_shards"),
    max_length=2048,
    model_config=model_config,
    resampled=False,
    shuffle_buffer=1,
)
collator = MultimodalCollator(tokenizer, passthrough_time_series=True)

### Run the forecasts

For each sample with a numeric target, the cell runs the same forward pass as training and captures the de-normalized quantiles `(H, V, Q)` from the forecast head. It computes the following metrics on valid target cells:

| Metric | Meaning |
|---|---|
| `mae`, `rmse` | Median forecast error in the series' original units |
| `smae` | MAE divided by the per-channel history scale, which makes errors comparable across datasets |
| `smae_mean`, `smae_last` | Scaled MAE of two naive baselines: history mean and last observed value |
| `pinball` | Mean scaled pinball loss over all quantiles, the training objective |
| `coverage80` | Fraction of targets inside the 0.1–0.9 quantile band (ideal ≈ 0.8) |

Samples whose target exceeds the head's 720 × 10 shape are skipped, and at most `MAX_PER_TASK` samples are scored per SciTS task so a single task does not dominate. Increase `N_EVAL` for a more stable estimate.

In [ ]:
from collections import Counter

import numpy as np

N_EVAL = 200
MAX_PER_TASK = 20
N_PLOT = 6


def last_observed(history, num_vars, fallback):
    """Last observed value per channel; history may carry an appended observation mask."""
    history = history[:, None] if history.dim() == 1 else history
    values = history[:, :num_vars]
    observed = torch.isfinite(values)
    if history.shape[1] == 2 * num_vars:
        observed &= history[:, num_vars:].bool()
    out = fallback.clone()
    for channel in range(num_vars):
        idx = observed[:, channel].nonzero()
        if len(idx):
            out[channel] = values[idx[-1, 0], channel]
    return out


captured = {}
hook = decoder.register_forward_hook(
    lambda module, args, output: captured.__setitem__("quantiles", output[0].detach().float().cpu())
)

records, examples, skipped = [], [], Counter()
task_counts = Counter()
try:
    with torch.inference_mode():
        for sample in val_dataset:
            if len(records) >= N_EVAL:
                break
            if sample.get("time_series_target") is None:
                skipped["no numeric target"] += 1
                continue
            task_id = sample.get("task_id", "unknown")
            if task_counts[task_id] >= MAX_PER_TASK:
                skipped["task already has MAX_PER_TASK samples"] += 1
                continue
            target = sample["time_series_target"].float()
            target = target[:, None] if target.dim() == 1 else target
            H, V = target.shape
            if H > HORIZON or V > NUM_VARS:
                skipped["target larger than head"] += 1
                continue
            mask = sample["time_series_target_mask"].reshape(H, V) & torch.isfinite(target)
            if not mask.any():
                skipped["no valid target cells"] += 1
                continue

            batch = {
                k: v.to(DEVICE) if isinstance(v, torch.Tensor)
                else [x.to(DEVICE) for x in v] if isinstance(v, list) and v and isinstance(v[0], torch.Tensor)
                else v
                for k, v in collator([sample]).items()
            }
            captured.clear()
            try:
                model(batch, labels=batch["text"])
            except RuntimeError as exc:
                skipped["forward error"] += 1
                print(f"skip {sample['_metadata']}: {exc}")
                continue

            q = captured["quantiles"][0, :H, :V]
            task_counts[task_id] += 1
            mean = sample["time_series_history_mean"][:V].float()
            scale = sample["time_series_history_scale"][:V].float()
            history = torch.as_tensor(sample["time_series"]).float()
            last = last_observed(history, V, mean)

            y = target[mask]
            scale_m = scale.expand(H, V)[mask]
            median = q[..., MEDIAN][mask]
            levels = torch.tensor(QUANTILES)
            scaled_err = (y[:, None] - q[mask]) / scale_m[:, None]
            records.append({
                "key": sample["_metadata"],
                "task_id": sample.get("task_id", "unknown"),
                "history_len": history.shape[0],
                "horizon": H,
                "variates": V,
                "valid_cells": int(mask.sum()),
                "mae": (median - y).abs().mean().item(),
                "rmse": (median - y).pow(2).mean().sqrt().item(),
                "smae": ((median - y).abs() / scale_m).mean().item(),
                "smae_mean": ((mean.expand(H, V)[mask] - y).abs() / scale_m).mean().item(),
                "smae_last": ((last.expand(H, V)[mask] - y).abs() / scale_m).mean().item(),
                "pinball": torch.maximum(levels * scaled_err, (levels - 1) * scaled_err).mean().item(),
                "coverage80": ((y >= q[..., 0][mask]) & (y <= q[..., -1][mask])).float().mean().item(),
            })
            if len(examples) < N_PLOT:
                values = history[:, None] if history.dim() == 1 else history
                examples.append({
                    "key": sample["_metadata"],
                    "task_id": records[-1]["task_id"],
                    "history": values[-max(3 * H, 64):, :V].numpy(),
                    "target": torch.where(mask, target, torch.nan).numpy(),
                    "quantiles": q.numpy(),
                })
            if len(records) % 25 == 0:
                print(f"{len(records)}/{N_EVAL} samples scored")
finally:
    hook.remove()

print(f"scored {len(records)} samples; skipped {dict(skipped)}")

### Results

The table gives macro averages over samples. `skill_vs_last` is `1 - smae / smae_last`, so positive values mean the model beats last-value persistence. The per-task table shows where the head helps and where it does not.

In [ ]:
import pandas as pd

results = pd.DataFrame(records)
metrics = ["mae", "rmse", "smae", "smae_mean", "smae_last", "pinball", "coverage80"]

overall = results[metrics].mean().to_frame(f"{EVAL_CHECKPOINT.name} (n={len(results)})").T
overall["skill_vs_last"] = 1 - overall["smae"] / overall["smae_last"]
display(overall.round(4))

per_task = results.groupby("task_id").agg(
    n=("key", "size"), horizon=("horizon", "median"), variates=("variates", "median"),
    **{m: (m, "mean") for m in ["smae", "smae_mean", "smae_last", "pinball", "coverage80"]},
)
per_task["skill_vs_last"] = 1 - per_task["smae"] / per_task["smae_last"]
display(per_task.sort_values("n", ascending=False).round(4))

### Example forecasts

Each panel shows channel 0 of one held-out sample: the tail of the history, the ground-truth target (masked cells omitted), the median forecast, and the 0.1–0.9 quantile band.

In [ ]:
fig, axes = plt.subplots(len(examples), 1, figsize=(11, 2.8 * len(examples)), squeeze=False)
for ax, ex in zip(axes[:, 0], examples, strict=True):
    hist, y, q = ex["history"], ex["target"], ex["quantiles"]
    t_hist, t_fut = np.arange(-len(hist), 0), np.arange(len(y))
    ax.plot(t_hist, hist[:, 0], color="black", lw=0.8, label="history")
    ax.plot(t_fut, y[:, 0], color="tab:blue", lw=1.0, label="target")
    ax.plot(t_fut, q[:, 0, MEDIAN], color="tab:orange", lw=1.0, label="median forecast")
    ax.fill_between(t_fut, q[:, 0, 0], q[:, 0, -1], color="tab:orange", alpha=0.25,
                    label=f"q{QUANTILES[0]}–q{QUANTILES[-1]}")
    ax.axvline(0, color="gray", ls=":", lw=0.8)
    ax.set_title(f"{ex['task_id']} | {ex['key']} | channel 0 of {y.shape[1]}", fontsize=9)
axes[0, 0].legend(loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

## Next steps

- Change `EVAL_CHECKPOINT` to `step_500` … `step_2000` (or later) and rerun section 4 to track held-out quality across training.
- Increase `N_EVAL` to cover all of `val_shards/` before you report numbers.
- Save `results` with `results.to_csv(...)` so you can compare checkpoints or presets.